# 攻撃的な投稿に対して落ち着いた返答を作る（カウンタースピーチ）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_dialogue/counter_speech.ipynb)

NLP クックブック 25-10 ／ 対話システム

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

差別的・攻撃的な投稿に対して、削除するだけでなく、冷静に反論したり、言われた側に寄り添ったりする返答を作りたい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。T4 なら 1 応答あたり数秒〜10 秒ほど (GPU メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=140, **kwargs):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

posts = ["女に管理職なんて無理に決まってる。感情的だからな。", "外国人が増えたせいで治安が悪くなった。全員出て行ってほしい。",
         "あの人、日本語うまいね。外国人なのに。", "年寄りは新しいことを覚えられないんだから、さっさと引退すればいいのに。"]

STRATEGIES = {
    "事実で反論": "事実や根拠を示して、決めつけが正しくないことを穏やかに指摘する",
    "問いかけ": "相手を責めずに、なぜそう思うのかを問いかけて、考え直すきっかけを作る",
    "当事者への共感": "この投稿で傷つく人の立場に立ち、その人たちを支える言葉をかける",
}
PROMPT = ("SNS に次の投稿がありました。これに対する返信を書いてください。\n方針: {strategy}。\n"
          "条件: 相手を侮辱しない。攻撃的な言葉を使わない。2 文以内で、丁寧な言葉で書く。返信の文だけを出力する。\n\n投稿: {post}")
for post in posts[:2]:   # 6 回生成する。T4 で 1〜2 分 (モデルの読み込みを除く)
    print("投稿:", post)
    for label, strategy in STRATEGIES.items():
        print(f"   [{label}] {chat(PROMPT.format(strategy=strategy, post=post))}")

投稿: 女に管理職なんて無理に決まってる。感情的だからな。
   [事実で反論] 管理職には感情のコントロールも重要なので、性別だけで判断するのは適切ではありません。能力や適性に基づいて判断すべきでしょう。
   [問いかけ] 女性が管理職に就くことに反対の意見ですね。確かに、管理職には論理的思考や決断力が求められる面があります。一方で、共感性やコミュニケーション能力も重要であると考えます。性別ではなく、個人としての能力や適性を重視すべきではないでしょうか。
   [当事者への共感] 女性全員が感情的であるとは限りませんし、管理職として活躍できる能力は性別ではなく個人によって異なります。あなたの意見は一部の傾向を一般化している可能性があります。誰もが平等にチャンスを持つべきだと考えます。応援しています。
投稿: 外国人が増えたせいで治安が悪くなった。全員出て行ってほしい。
   [事実で反論] 外国人の増加が治安悪化の原因であると断定するには、より詳細な分析が必要です。一般的に、犯罪率は様々な要因によって左右され、外国人人口のみが原因であるとは限りません。例えば、経済状況、教育水準、法整備などが影響します。穏やかな環境を維持するためにも、建設的な議論と対策を考えることが重要です。
   [問いかけ] 外国人の増加と治安の悪化にはさまざまな要因があり、単純な因果関係とは限りません。その地域で起きた具体的な出来事や統計データなどをもとに、議論を深めてみてはいかがでしょうか。
   [当事者への共感] 外国人が増えたことで不安を感じている気持ち、理解できます。しかし、多様性は私たちの社会を豊かにする要素でもあります。もし何か困ったことがあれば、いつでも助けますので、安心して生活できる方法を一緒に考えましょう。


In [2]:
# はっきりした暴言ではないが、相手を傷つける発言 (マイクロアグレッション) への返答
for post in posts[2:]:
    print("投稿:", post)
    print("   ", chat(PROMPT.format(strategy="悪気がないかもしれないことを踏まえつつ、その言い方がなぜ相手を傷つけうるのかを、やさしく説明する", post=post)))

投稿: あの人、日本語うまいね。外国人なのに。
    その言い方は、相手の努力や背景を尊重していないと受け取られかねません。外国語を上手に話すことは、その人の努力や能力を示すものであり、外国人であることを強調するのは適切ではありません。相手の文化的背景を認め、その言語スキルを評価する言い方の方がよいでしょう。例えば、「その日本語、とても流暢ですね。練習の成果でしょうか」などいかがでしょうか。
投稿: 年寄りは新しいことを覚えられないんだから、さっさと引退すればいいのに。
    お年寄りの方々は長年の経験と知識をお持ちです。新しいことを覚えるスピードは人それぞれであり、引退を強制するような発言は尊重を欠いています。お互いを理解し、尊重することが大切です。


In [3]:
# 生成した返答を機械的に点検する: 返答自体が攻撃的になっていないか、根拠のない数字や事実を作っていないか
import re

NG = ["バカ", "馬鹿", "アホ", "クズ", "黙れ", "お前", "頭が悪い", "恥を知れ", "差別主義者"]
def check(reply):
    return {"攻撃的な語": [w for w in NG if w in reply], "数字 (出典の確認が必要)": re.findall(r"\d+[%％割倍]|\d{4} 年", reply), "長さ": len(reply)}

for post in posts[:2]:
    reply = chat(PROMPT.format(strategy=STRATEGIES["事実で反論"], post=post))
    print(reply, "\n   →", check(reply))

管理職には感情のコントロールも重要なので、性別だけで判断するのは適切ではありません。能力や適性に基づいて判断すべきでしょう。 
   → {'攻撃的な語': [], '数字 (出典の確認が必要)': [], '長さ': 62}
外国人の増加が治安悪化の原因であると断定するには、より詳細な分析が必要です。一般的に、犯罪率は様々な要因によって左右され、外国人人口のみが原因であるとは限りません。例えば、経済状況、教育水準、法整備などが影響します。穏やかな環境を維持するためにも、建設的な議論と対策を考えることが重要です。 
   → {'攻撃的な語': [], '数字 (出典の確認が必要)': [], '長さ': 145}


- カウンタースピーチは、有害な投稿を削除するのではなく、言葉で対抗する取り組み。事実を示す、矛盾を指摘する、問いかける、当事者に共感を示す、などの方針がある。どれが有効かは場面による。
- 明確な暴言だけでなく、悪意のない決めつけや「ほめ言葉」の形をとった偏見（マイクロアグレッション）も対象になる。こちらは、相手を責めずに気付きを促す返答が向く。
- LLM は、もっともらしい統計や研究結果を作ってしまうことがある。事実で反論する返答は、根拠を人が確認してから使う。確認できない数字は入れさせない。
- 小さなモデルの出力には、実際に次のような失敗が出た: 「2 文以内」の条件を守らずに 4 文書く、当事者に寄り添う方針なのに、投稿者本人に向けて「あなたの意見は一部の傾向を一般化している」「応援しています」と書いてしまう。英単語が混ざる（「peacefulな」のような）失敗も起こりうる。方針が守られているかは、生成のたびに確かめる必要がある。
- 返答が説教調・上から目線になると逆効果になる。生成した返答が攻撃的でないかを点検し、最終的には人が読んでから投稿する。自動で返信する仕組みにはしない。
- モデルによっては、有害な投稿を入力に含むだけで応答を拒否することがある。その場合は、目的（有害表現への対抗）を指示の中で明確にする。